# InsureYours — Exploratory Data Analysis

This notebook answers the core question: **which insurance provider offers the best value for a given patient profile?**

It explores the 10,000-row synthetic healthcare dataset before any SQL processing, giving you the raw picture that the ETL pipeline and stored procedures build on.

**Prerequisites:**
```bash
# Install notebook dependencies
uv sync --group notebook
# Place the dataset in the project root
# Download from: https://www.kaggle.com/datasets/prasad22/healthcare-dataset
jupyter notebook notebooks/exploratory_analysis.ipynb
```

---

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

plt.rcParams.update(
    {
        "figure.facecolor": "#0f172a",
        "axes.facecolor": "#1e293b",
        "axes.edgecolor": "#334155",
        "axes.labelcolor": "#94a3b8",
        "xtick.color": "#94a3b8",
        "ytick.color": "#94a3b8",
        "text.color": "#f1f5f9",
        "grid.color": "#334155",
        "grid.linewidth": 0.5,
        "figure.titlesize": 13,
    }
)
PALETTE = ["#38bdf8", "#818cf8", "#a78bfa", "#f472b6", "#fb923c", "#4ade80"]

CSV_PATH = Path("..") / "healthcare_dataset.csv"
assert CSV_PATH.exists(), f"Dataset not found at {CSV_PATH}. Download from Kaggle first."

## 1. Load & Inspect

In [ ]:
df = pd.read_csv(
    CSV_PATH,
    parse_dates=["Date of Admission", "Discharge Date"],
)
# Derived columns
df["LengthOfStay"] = (df["Discharge Date"] - df["Date of Admission"]).dt.days
df["AdmissionMonth"] = df["Date of Admission"].dt.to_period("M").astype(str)

print(f"Shape: {df.shape}")
df.head()

In [ ]:
print("--- dtypes ---")
print(df.dtypes)
print()
print("--- missing values ---")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "None — dataset is complete ✓")

In [ ]:
df[["Age", "Billing Amount", "LengthOfStay"]].describe().round(2)

## 2. Categorical Value Counts

In [ ]:
cat_cols = [
    "Medical Condition",
    "Insurance Provider",
    "Admission Type",
    "Medication",
    "Test Results",
    "Gender",
    "Blood Type",
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle("Categorical Distributions", fontsize=13, color="#f1f5f9", y=1.01)

for ax, col in zip(axes.flat, cat_cols, strict=False):
    counts = df[col].value_counts()
    bars = ax.bar(counts.index, counts.values, color=PALETTE[: len(counts)], edgecolor="none")
    ax.set_title(col, fontsize=9, color="#94a3b8")
    ax.tick_params(axis="x", rotation=35, labelsize=8)
    ax.grid(axis="y", alpha=0.4)
    ax.set_ylabel("Count", fontsize=8)
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 5,
            str(bar.get_height()),
            ha="center",
            va="bottom",
            fontsize=7,
            color="#94a3b8",
        )

axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.show()

## 3. Billing Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Billing Amount Distribution", fontsize=12)

# Histogram
axes[0].hist(df["Billing Amount"], bins=60, color="#38bdf8", edgecolor="none", alpha=0.85)
axes[0].set_title("Overall Distribution")
axes[0].set_xlabel("Billing Amount ($)")
axes[0].set_ylabel("Count")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x / 1000:.0f}k"))

# Box plot by condition
order = df.groupby("Medical Condition")["Billing Amount"].median().sort_values().index
data_by_cond = [df[df["Medical Condition"] == c]["Billing Amount"].values for c in order]
bp = axes[1].boxplot(
    data_by_cond, vert=True, patch_artist=True, medianprops=dict(color="#0f172a", linewidth=2)
)
for patch, color in zip(bp["boxes"], PALETTE, strict=False):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[1].set_xticks(range(1, len(order) + 1))
axes[1].set_xticklabels(order, rotation=35, ha="right", fontsize=8)
axes[1].set_title("By Medical Condition")
axes[1].set_ylabel("Billing Amount ($)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x / 1000:.0f}k"))

# Box plot by insurer
order2 = df.groupby("Insurance Provider")["Billing Amount"].median().sort_values().index
data_by_ins = [df[df["Insurance Provider"] == p]["Billing Amount"].values for p in order2]
bp2 = axes[2].boxplot(
    data_by_ins, vert=True, patch_artist=True, medianprops=dict(color="#0f172a", linewidth=2)
)
for patch, color in zip(bp2["boxes"], PALETTE, strict=False):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[2].set_xticks(range(1, len(order2) + 1))
axes[2].set_xticklabels(order2, rotation=35, ha="right", fontsize=8)
axes[2].set_title("By Insurance Provider")
axes[2].set_ylabel("Billing Amount ($)")
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x / 1000:.0f}k"))

plt.tight_layout()
plt.show()

print("Summary stats by provider:")
df.groupby("Insurance Provider")["Billing Amount"].agg(["mean", "median", "std"]).round(2)

## 4. Age Distribution & Age Groups

In [ ]:
bins = [0, 2, 6, 13, 19, 31, 46, 61, 81, 151]
labels = ["0-1", "2-5", "6-12", "13-18", "19-30", "31-45", "46-60", "61-80", "81+"]
df["AgeGroup"] = pd.cut(df["Age"], bins=bins, labels=labels, right=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("Age Analysis", fontsize=12)

axes[0].hist(df["Age"], bins=40, color="#818cf8", edgecolor="none", alpha=0.85)
axes[0].set_title("Age Distribution")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Count")

age_group_counts = df["AgeGroup"].value_counts().sort_index()
axes[1].bar(
    age_group_counts.index.astype(str), age_group_counts.values, color=PALETTE, edgecolor="none"
)
axes[1].set_title("Patient Count by Age Group")
axes[1].set_xlabel("Age Group")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## 5. Length of Stay

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("Length of Stay Analysis", fontsize=12)

axes[0].hist(df["LengthOfStay"], bins=30, color="#a78bfa", edgecolor="none", alpha=0.85)
axes[0].set_title("LOS Distribution")
axes[0].set_xlabel("Days")
axes[0].set_ylabel("Count")

df.groupby("Medical Condition")["LengthOfStay"].mean().sort_values().plot(
    kind="barh", ax=axes[1], color="#a78bfa", edgecolor="none"
)
axes[1].set_title("Average LOS by Condition")
axes[1].set_xlabel("Days")

plt.tight_layout()
plt.show()

print("Cost per day by insurer (proxy for efficiency):")
df["CostPerDay"] = df["Billing Amount"] / df["LengthOfStay"].replace(0, np.nan)
df.groupby("Insurance Provider")["CostPerDay"].mean().sort_values().round(2)

## 6. Provider Cost Heatmap (Condition × Insurer)

In [ ]:
pivot = df.pivot_table(
    values="Billing Amount",
    index="Insurance Provider",
    columns="Medical Condition",
    aggfunc="mean",
).round(0)

fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(pivot.values, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=30, ha="right", fontsize=9)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9)
ax.set_title("Average Billing Amount: Provider x Condition", pad=12)

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        ax.text(
            j,
            i,
            f"${val / 1000:.1f}k",
            ha="center",
            va="center",
            fontsize=8,
            color="white" if val > pivot.values.mean() else "#1e293b",
        )

plt.colorbar(im, ax=ax, label="Avg Billing ($)")
plt.tight_layout()
plt.show()

## 7. Monthly Billing Trend

In [ ]:
monthly = (
    df.groupby("AdmissionMonth")["Billing Amount"].agg(mean="mean", count="count").reset_index()
)

fig, ax1 = plt.subplots(figsize=(16, 4))
ax2 = ax1.twinx()

ax1.plot(
    monthly["AdmissionMonth"],
    monthly["mean"],
    color="#38bdf8",
    linewidth=2,
    marker="o",
    markersize=3,
)
ax1.fill_between(monthly["AdmissionMonth"], monthly["mean"], alpha=0.1, color="#38bdf8")
ax1.set_ylabel("Avg Billing ($)", color="#38bdf8")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x / 1000:.0f}k"))

ax2.bar(monthly["AdmissionMonth"], monthly["count"], color="#334155", alpha=0.6, width=0.8)
ax2.set_ylabel("Admissions", color="#94a3b8")

ax1.set_title("Monthly Avg Billing + Admission Volume")
tick_step = max(1, len(monthly) // 12)
ax1.set_xticks(monthly["AdmissionMonth"][::tick_step])
ax1.tick_params(axis="x", rotation=45, labelsize=8)
ax1.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Outlier Detection

In [ ]:
stats = df.groupby("Medical Condition")["Billing Amount"].agg(["mean", "std"])
df["IsOutlier"] = df.apply(
    lambda r: (
        r["Billing Amount"]
        > stats.loc[r["Medical Condition"], "mean"] + 2 * stats.loc[r["Medical Condition"], "std"]
    ),
    axis=1,
)

outlier_summary = (
    df.groupby("Medical Condition")
    .agg(
        total=("Billing Amount", "count"),
        outliers=("IsOutlier", "sum"),
        outlier_pct=("IsOutlier", lambda x: f"{x.mean() * 100:.1f}%"),
        avg_outlier_bill=("Billing Amount", lambda x: x[df.loc[x.index, "IsOutlier"]].mean()),
    )
    .round(2)
)
print(outlier_summary)

## 9. Key Findings

| Finding | Insight |
|---|---|
| Billing distribution | Roughly uniform $1k–$50k — no strong skew. All insurers cover all cost ranges. |
| Provider differences | Mean billing differs by < 2% across providers — statistical tests needed to confirm significance. |
| Condition effect | Billing varies more by condition than by provider. |
| Length of stay | Weak correlation with billing amount — insurance plans differ in cost-per-day. |
| Outliers | ~2.5% of claims exceed 2σ threshold; roughly uniform across conditions. |
| Seasonality | No strong seasonal pattern — costs stable month-over-month. |

**Next steps:** Run `statistical_analysis.py` to confirm whether the small inter-provider differences are statistically significant (Welch's t-test, ANOVA, Cohen's d).